# Getting Started with Latent Variable Models (Demo with Simulated Data)

In this tutorial, we will demonstrate how to define a latent variable model with two outputs using the {py:class}`~pollux.models.LVM` class. This tutorial is somewhat of a companion to [Intro to Latent Variable Models](LVM-math-notes.ipynb), which covers the mathematical basis and framing of what an LVM is. Here, we instead focus on the workflow within Pollux and on choices you have to make when setting up and using a model (which optimizer to use, how many latent dimensions to give the model, and whether to preprocess your data). `LVM` is the general framework in Pollux: you choose a latent dimensionality and compose transforms from the latents to each observed output. (Pollux also comes with predefined architectures built on it, like {py:class}`~pollux.models.Lux` and {py:class}`~pollux.models.Cannon`, which have their own tutorials focused on stellar spectroscopy.) 

We will use simulated data for this tutorial, but the outputs are meant to loosely represent (1) stellar labels, like element abundances, stellar parameters, etc., and (2) stellar spectra (fluxes) on a wavelength-aligned grid of pixels. For the model, we will use a bi-linear structure in which both outputs are generated as linear transformations of a latent representation of each star. We will use a latent dimensionality that is larger than the number of stellar labels but smaller than the number of pixels in the spectra.

We will start with some standard imports and set up the simulated data.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.distributions.constraints import real

import pollux as plx
from pollux.models.transforms import AffineTransform, LinearTransform

jax.config.update("jax_enable_x64", True)
%matplotlib inline

## Generating simulated data

We will generate data for 2048 stars, with a latent dimensionality of 8, but only 2 labels and 128 pixels in the spectra. We will define the linear transformations that generate the simulated labels and spectra to have some strict structure: the first 2 latent dimensions will be used to generate the labels, and will correlate with the strength of Gaussian "spectral lines" in the simulated spectra. This is purely for demonstration purposes, and we could instead have used random linear transformations (e.g., with all elements of the transform matrices drawn from a Normal or uniform distribution).

We then put the two labels into physically meaningful units, so that they look like real stellar labels: the first is effective-temperature-like, in Kelvin, and the second is abundance-like, in dex. This matters more than it might seem --- we come back to it in the section on preprocessing below.

In [ ]:
from helpers import make_simulated_linear_data

n_stars = 2048  # number of simulated stars to generate in the train and test sets
n_latents = 8  # size of the latent vector per star
n_labels = 2  # number of labels to generate per star
n_flux = 128  # number of spectral flux pixels per star

rng = np.random.default_rng(seed=8675309)

A = np.zeros((n_labels, n_latents))
A[0, 0] = 1.0
A[1, 1] = 1.0

B = rng.normal(scale=0.1, size=(n_flux, n_latents))
B[:, 0] = B[:, 0] + 4 * np.exp(-0.5 * (np.arange(n_flux) - n_flux / 2) ** 2 / 5**2)
B[:, 1] = B[:, 1] + 2 * np.exp(-0.5 * (np.arange(n_flux) - n_flux / 4) ** 2 / 3**2)

data, truth = make_simulated_linear_data(
    n_stars=n_stars,
    n_latents=n_latents,
    n_flux=n_flux,
    n_labels=n_labels,
    A=A,
    B=B,
    rng=rng,
)

# Put the labels into physically meaningful units: one Teff-like (K), one
# abundance-like (dex). This is just a linear reparameterization of the rows of A,
# but it makes the two labels differ in scale by four orders of magnitude.
label_loc = np.array([5000.0, 0.0])
label_scale = np.array([500.0, 0.3])

data["label"] = data["label"] * label_scale + label_loc
data["label_err"] = data["label_err"] * label_scale
truth["label"] = truth["label"] * label_scale + label_loc

The data have simulated uncertainties, which the {py:class}`~pollux.models.LVM` will use to define the likelihoods for the labels and spectra. Here are a few examples of the simulated spectra, ordered by (and colored by) the value of the first (0th index) label:

In [ ]:
cmap = plt.get_cmap("coolwarm")
norm = mpl.colors.Normalize(
    vmin=data["label"][:, 0].min(), vmax=data["label"][:, 0].max()
)
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

idx = np.argsort(data["label"][:, 0])
for i in np.linspace(0, len(idx) - 1, 16).astype(int):
    ax.plot(
        data["flux"][idx[i]],
        marker="",
        drawstyle="steps-mid",
        color=cmap(norm(data["label"][idx[i], 0])),
    )
ax.set(xlabel="pixel (wavelength)", ylabel="flux", title="Simulated spectra (flux)")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cb = fig.colorbar(sm, ax=ax)
cb.set_label("Label 0 value")

To use this data with the {py:class}`~pollux.models.LVM`, we will need to define a {py:class}`~pollux.data.PolluxData` instance. This object acts as a container for the data and uncertainties, and also provides a mechanism to define "pre-processors" for the data. In this case, we will define pre-processors that normalize the labels and spectra to have zero mean and unit variance:

In [ ]:
all_data = plx.data.PolluxData(
    flux=plx.data.OutputData(
        data["flux"],
        err=data["flux_err"],
        preprocessor=plx.data.ShiftScalePreprocessor.from_data(data["flux"]),
    ),
    label=plx.data.OutputData(
        data["label"],
        err=data["label_err"],
        preprocessor=plx.data.ShiftScalePreprocessor.from_data(data["label"]),
    ),
).preprocess()

Note that we keep only the *preprocessed* container. Each {py:class}`~pollux.data.OutputData` carries its preprocessor with it, and slicing preserves it, so the data in natural units are always one `.unprocess()` away. There is no need to hold on to a second, unprocessed copy — and keeping one invites passing the wrong container to `.optimize()`, which fits the raw values against priors that assume the preprocessed scale. (Pollux warns if you do.)

For this example, we will use the model in a "supervised" or "train and apply" mode, in which we will train the model on a subset of the data and then apply it to the remaining data. We don't have to do this: we could instead fit the model to all of the data at once. But this is a more realistic scenario for real data, in which we have some stars with labels and spectra and some stars with only spectra. We will use the first 1024 stars for training and the remaining 1024 stars for testing (since they are not ordered in any way):

In [ ]:
train_data = all_data[: n_stars // 2]
test_data = all_data[n_stars // 2 :]
len(train_data), len(test_data)

## Constructing the model

We create a model by first defining a {py:class}`~pollux.models.LVM` instance with a specified latent dimensionality. In this case, we know that the data were generated with a latent dimensionality of 8, so we will use that value initially in this example:

In [ ]:
model = plx.LVM(latent_size=8)

We then have to tell the model about the outputs (i.e. predict data) using the {py:meth}`~pollux.models.LVM.register_output` method. For this method, we specify an output name and a transform that specifies how the output will be generated from the latent representation. We currently have a few built-in transforms (such as {py:class}`~pollux.models.transforms.LinearTransform`, {py:class}`~pollux.models.transforms.AffineTransform`, and {py:class}`~pollux.models.transforms.PolyFeatureTransform`), but note that it is possible to define custom transforms by subclassing the {py:class}`~pollux.models.transforms.AbstractTransform` class.

In this example, we will use linear transformations (using {py:class}`~pollux.models.transforms.LinearTransform`) for both outputs of our demo model. We can define the transforms by, at minimum, specifying the output dimensionality for each output. In this case, the output names should match the names of the blocks in the data:

In [ ]:
print(all_data.keys())
model.register_output("label", LinearTransform(output_size=n_labels))
model.register_output("flux", LinearTransform(output_size=n_flux))

With no other arguments, the {py:class}`~pollux.models.transforms.LinearTransform` will generate a linear transformation matrix and use a {py:class}`~numpyro.distributions.continuous.Normal` prior for the elements of the matrix with zero mean and unit variance. When optimizing, this is equivalent to placing an L2 regularization on the elements of the matrix --- see the [objective written out in the intro note](LVM-math-notes.ipynb) for where the prior enters. However, we can also override the default prior by specifying the `priors` argument to the {py:class}`~pollux.models.transforms.LinearTransform` initializer. This argument should be a dictionary with keys that match the names of the parameters in the transform and values that are instances of {py:class}`~numpyro.distributions.distribution.Distribution`. For the {py:class}`~pollux.models.transforms.LinearTransform`, the parameter name is `A` and this represents the matrix that maps latent dimensionality to output dimensionality.

For example, if we wanted to use an L2 regularization with a different regularization strength `alpha`, we could specify a different prior for `A`:

In [ ]:
alpha = 100.0
trans = LinearTransform(
    output_size=n_labels, priors={"A": dist.Normal(0.0, jnp.sqrt(1 / alpha))}
)

Or, to disable regularization entirely, we could instead specify a {py:class}`~numpyro.distributions.distribution.ImproperUniform` prior:

In [ ]:
trans = LinearTransform(
    output_size=n_labels, priors={"A": dist.ImproperUniform(real, (), ())}
)

For this example, we will proceed with the default priors for the linear transformation matrix elements.

As an initial test of the using the model, we will generate random values for latent vectors and the linear transform parameters and use the {py:meth}`~pollux.models.LVM.predict_outputs` method to generate predictions for the labels and spectra. These predictions will be meaningless in practice, because we have not yet optimized the parameters of the model, but they will demonstrate the structure of the model:

In [ ]:
rngs = jax.random.split(jax.random.PRNGKey(42), 3)

# For this demo, we'll generate outputs for 10 objects
latents = jax.random.normal(rngs[0], shape=(10, model.latent_size))
pars = {
    "label": {
        "data": {"A": jax.random.normal(rngs[1], shape=(n_labels, model.latent_size))}
    },
    "flux": {
        "data": {"A": jax.random.normal(rngs[2], shape=(n_flux, model.latent_size))}
    },
}
outputs = model.predict_outputs(pars, latents)
outputs["label"].shape, outputs["flux"].shape

Later, once we have optimized the parameters of the model, we can use this method to generate predictions for new or held-out data, or to validate the model.

## Optimizing the model with training data (i.e. training the model)

As mentioned above, in this demonstration, we will use the first 1024 stars for training the model and the remaining 1024 stars for testing the model performance. We will optimize the model parameters using the training data and then evaluate the model on the test data and compare with the true values.

Both outputs of this model are linear in the latents, and that structure is worth exploiting. Holding the transformation matrices fixed, solving for the latents is a weighted linear least-squares problem with an exact solution; holding the latents fixed, solving for the matrices is too. {py:meth}`~pollux.models.LVM.optimize_iterative` alternates between those exact solves, which needs no learning rate and no step count:

In [ ]:
trained = model.optimize_iterative(
    train_data,
    max_cycles=64,
    rng_key=jax.random.PRNGKey(112358),
    progress=False,
)
opt_pars = trained.params

print(f"converged: {trained.converged} after {trained.n_cycles} cycles")

Pollux does not take our word for it that the model is linear: it works out which blocks of parameters admit a closed-form solve by linearizing the transforms and testing them, and falls back to gradient descent for any block that does not qualify. We can see what each block actually got:

In [ ]:
[(block.name, block.optimizer) for block in trained.blocks]

All three blocks came back as `least_squares`, meaning nothing in this fit was approximated by gradient descent. (The [Linearized, closed-form solves](../linear-solves.md) note describes how that determination is made, and why it is done by measurement rather than by checking the type of each transform.)

The loss here is recorded once per cycle rather than once per gradient step:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.plot(trained.losses_per_cycle, marker="o", color="k")
_ = ax.set(xlabel="cycle", ylabel="loss", title="Iterative optimization")

The loss drops sharply over the first few cycles and then flattens, and `optimize_iterative` stops on its own once the relative change falls below its tolerance --- there is no step count to tune.

### When you need the general-purpose optimizer

{py:meth}`~pollux.models.LVM.optimize` takes a different approach: it hands the whole model to `numpyro` and runs stochastic variational inference with an `AutoDelta` guide, which is a roundabout way of finding the same maximum *a posteriori* parameters by gradient descent. It works for any model, but it has to be told a learning rate and a number of steps. Let's run it on the same problem and compare:

In [ ]:
t0 = time.time()
_ = model.optimize_iterative(
    train_data, max_cycles=64, rng_key=jax.random.PRNGKey(112358), progress=False
)
iterative_time = time.time() - t0

# warm up so that both are timed after JAX has compiled, not one of each
_ = model.optimize(
    train_data,
    rng_key=jax.random.PRNGKey(112358),
    optimizer=numpyro.optim.Adam(1e-3),
    num_steps=10,
    svi_run_kwargs={"progress_bar": False},
)

t0 = time.time()
svi_pars, svi_results = model.optimize(
    train_data,
    rng_key=jax.random.PRNGKey(112358),
    optimizer=numpyro.optim.Adam(1e-3),
    num_steps=10_000,
    svi_run_kwargs={"progress_bar": False},
)
svi_results.losses.block_until_ready()
svi_time = time.time() - t0

print(f"{'method':<22} {'time (s)':>9} {'final loss':>14}")
print(
    f"{'optimize_iterative':<22} {iterative_time:9.2f} {trained.losses_per_cycle[-1]:14.1f}"
)
print(
    f"{'optimize (Adam, 10k)':<22} {svi_time:9.2f} {float(svi_results.losses[-1]):14.1f}"
)

For a model like this one the closed-form path wins on both counts, and the gradient-based run has not even finished converging --- its loss was still decreasing when the step budget ran out, so it would need more steps, a larger learning rate, or both.

So why keep {py:meth}`~pollux.models.LVM.optimize` at all? Because the closed-form solves only exist for blocks that are linear in the parameters being solved for. Reach for `optimize` when:

- a transform is nonlinear in the latents --- a polynomial expansion, a neural network, a Gaussian process. `optimize_iterative` will still use exact solves for whichever blocks qualify and fall back to gradient descent for the rest, warning you when it does, but a model that is nonlinear throughout has nothing left to exploit.
- you want something other than a point estimate. `optimize` accepts any `numpyro` autoguide, so switching to a `AutoNormal` guide, or handing the model to MCMC, starts here.

In [ ]:
opt_pars["label"], opt_pars["flux"]

As we saw above, we can use the {py:meth}`~pollux.models.LVM.predict_outputs` method to generate predictions for the training data given the optimized latent vectors:

In [ ]:
# optimize() returns the latents inside the parameters, so they need not be passed
predict_train_values = model.predict_outputs(opt_pars)

In [ ]:
pt_style = {"ls": "none", "ms": 2.0, "alpha": 0.5, "marker": "o", "color": "k"}

fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for i in range(predict_train_values["label"].shape[1]):
    axes[i].plot(
        predict_train_values["label"][:, i], train_data["label"].data[:, i], **pt_style
    )
    axes[i].set(xlabel=f"Predicted label {i}", ylabel=f"True label {i}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle("Training set: predicted vs. true labels", fontsize=22)

In [ ]:
# Pick some random pixel values to compare:
pixel_idx = np.array([9, 49, 55, 80])

fig, axes = plt.subplots(
    1, len(pixel_idx), figsize=(4 * len(pixel_idx), 4), layout="constrained"
)
for i, j in enumerate(pixel_idx):
    axes[i].plot(
        predict_train_values["flux"][:, j], train_data["flux"].data[:, j], **pt_style
    )
    axes[i].set(xlabel=f"Predicted flux {j}", ylabel=f"True flux {j}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle(
    f"Training set: predicted vs. true flux ({len(pixel_idx)} random pixels)",
    fontsize=22,
)

Both the predicted training set labels and fluxes look reasonable, which is a good sign that the model has learned the structure of the data. However, we also want to evaluate the model on the test data, which the model has not seen during optimization.


## Optimize for latents for test set

To predict labels and fluxes for the test set, we need to optimize the latent vectors for the test set. When we do this, we want to hold fixed the linear transformation matrices we learned from the training set data. The {py:meth}`~pollux.models.LVM.output_pars` method selects exactly the parameters that carry over from one set of objects to another --- everything except the per-object latents:

In [ ]:
# Everything except the per-object latents, which is exactly what carries over
# from the training set to the test set:
fixed_pars = model.output_pars(opt_pars)

We then pass those as `fixed_pars` and restrict the optimization to the latents with `blocks=["latents"]`, so nothing else moves:

In [ ]:
test_result = model.optimize_iterative(
    test_data,
    blocks=["latents"],
    fixed_pars=fixed_pars,
    max_cycles=64,
    progress=False,
)
test_opt_pars = test_result.params

The returned parameters contain the fixed transformation matrices merged with the newly optimized latent vectors, so the result is a complete parameter set ready to predict with:

In [ ]:
sorted(test_opt_pars)

In [ ]:
test_opt_pars["latents"].shape

We can then use these latent vectors with the linear transformation matrices we learned from the training set data to generate predictions for the test set data:

In [ ]:
predict_test_values = model.predict_outputs(
    fixed_pars, latents=test_opt_pars["latents"]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for i in range(predict_test_values["label"].shape[1]):
    axes[i].plot(
        predict_test_values["label"][:, i], test_data["label"].data[:, i], **pt_style
    )
    axes[i].set(xlabel=f"Predicted label {i}", ylabel=f"True label {i}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle("Test set: predicted vs. true labels", fontsize=22)

In [ ]:
fig, axes = plt.subplots(
    1, len(pixel_idx), figsize=(4 * len(pixel_idx), 4), layout="constrained"
)
for i, j in enumerate(pixel_idx):
    axes[i].plot(
        predict_test_values["flux"][:, j], test_data["flux"].data[:, j], **pt_style
    )
    axes[i].set(xlabel=f"Predicted flux {j}", ylabel=f"True flux {j}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle(
    f"Test set: predicted vs. true flux ({len(pixel_idx)} random pixels)",
    fontsize=22,
)

Good, the model still seems to be working well on the test set data. We can now quantitatively evaluate the model performance by comparing the predicted labels and spectra with the true values.

The raw, predicted labels and spectra are in the pre-processed domain, so we need to apply the inverse transform of the pre-processors to get the predicted labels and spectra in the original data domain (to compare to the true data):

In [ ]:
predict_test_unprocessed = test_data.unprocess(predict_test_values)

The prediction error for the labels:

In [ ]:
np.std(predict_test_unprocessed["label"].data - truth["label"][n_stars // 2 :], axis=0)

And the mean prediction error for the fluxes (across all pixels):

In [ ]:
np.mean(
    np.std(
        predict_test_unprocessed["flux"].data - truth["flux"][n_stars // 2 :], axis=0
    )
)

## Optimize for latents with partial data

In real world cases with spectroscopic data, we will likely want to instead use the model to predict labels for sources that only have spectra and not labels. In this case, we can optimize the latent vectors for the test set data using only the spectra and not the labels.

To do this, we simply pass a {py:class}`~pollux.data.PolluxData` that contains only the flux. There is no need to tell the model which outputs to use: an output with no data contributes nothing to the likelihood, and {py:meth}`~pollux.models.LVM.optimize_iterative` works out which outputs the data actually carry.

In [ ]:
flux_only_data = plx.data.PolluxData(flux=test_data["flux"])

In [ ]:
test_result_flux = model.optimize_iterative(
    flux_only_data,
    blocks=["latents"],
    fixed_pars=fixed_pars,
    max_cycles=64,
    progress=False,
)
test_opt_pars_flux = test_result_flux.params

In [ ]:
predict_test_values_flux = model.predict_outputs(
    fixed_pars, latents=test_opt_pars_flux["latents"]
)
predict_test_unprocessed_flux = test_data.unprocess(predict_test_values_flux)

We now have optimized latent vectors for the test set data using only the spectral flux data. We can now compare the predict labels with the true labels:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for i in range(predict_test_values_flux["label"].shape[1]):
    axes[i].plot(
        predict_test_values_flux["label"][:, i],
        test_data["label"].data[:, i],
        **pt_style,
    )
    axes[i].set(xlabel=f"Predicted label {i}", ylabel=f"True label {i}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle(
    "Test set: predicted vs. true labels (using flux to predict)", fontsize=22
)

In [ ]:
np.std(
    predict_test_unprocessed_flux["label"].data - truth["label"][n_stars // 2 :], axis=0
)

Compared to the case above where we used both the labels and spectra to optimize the latent vectors for the test set, the prediction error is _slightly_ higher when using only the spectra, as we would expect: the model has less information to constrain the latent vectors.

## Choosing the latent dimensionality

So far we have used `latent_size=8` because that is how the data were generated. However, with real data, we won't know a priori what a good latent dimensionality is. This is the most consequential choice in setting up one of these linear LVMs, so it is worth seeing what happens when we get it wrong in either direction.

We will refit the model for a range of latent sizes, and for each one apply it to the held-out test set exactly as above:

In [ ]:
latent_sizes = [2, 4, 6, 8, 12, 16, 24, 32]

sweep_train_loss = []
sweep_test_rmse = []

for L in latent_sizes:
    _model = plx.LVM(latent_size=L)
    _model.register_output("label", LinearTransform(output_size=n_labels))
    _model.register_output("flux", LinearTransform(output_size=n_flux))

    _trained = _model.optimize_iterative(
        train_data, max_cycles=64, rng_key=jax.random.PRNGKey(112358), progress=False
    )
    _fixed = _model.output_pars(_trained.params)

    # apply to the test set using the spectra only, as we did above
    _applied = _model.optimize_iterative(
        flux_only_data, blocks=["latents"], fixed_pars=_fixed, progress=False
    )
    _pred = _model.predict_outputs(_fixed, latents=_applied.params["latents"])
    _pred_labels = test_data.unprocess(_pred)["label"].data

    sweep_train_loss.append(_trained.losses_per_cycle[-1])
    sweep_test_rmse.append(
        np.sqrt(
            np.mean(
                (np.asarray(_pred_labels) - truth["label"][n_stars // 2 :]) ** 2, axis=0
            )
        )
    )

sweep_test_rmse = np.array(sweep_test_rmse)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")

axes[0].plot(latent_sizes, sweep_train_loss, marker="o", color="k")
axes[0].set(xlabel="latent size", ylabel="final training loss", title="Training loss")

for i in range(n_labels):
    axes[1].plot(
        latent_sizes,
        sweep_test_rmse[:, i] / sweep_test_rmse[:, i].min(),
        marker="o",
        label=f"label {i}",
    )
axes[1].set(
    xlabel="latent size",
    ylabel="test RMSE / best",
    title="Held-out prediction error",
    yscale="log",
)
axes[1].legend()

for ax in axes:
    ax.axvline(n_latents, color="tab:green", ls="--", lw=1, zorder=-10)
_ = fig.suptitle("dashed line: the true latent dimensionality", fontsize=14)

The training loss decreases monotonically with the latent size. It always will: more latent dimensions can only fit the training data better, so this curve has no minimum and cannot tell you when to stop. Choosing the latent size by looking at the training loss will always tell you to use more.

The held-out test error has a clear minimum at the true dimensionality. Below it, the model has too few directions to describe the data and underfits badly. Above it, the extra directions have nothing real to describe and start absorbing noise from the training set, which does not generalize. Only data the model has not seen can distinguish these two failure modes. (The same logic drives the hyperparameter sweep in the [*Lux* tutorial](Lux-getting-started-apogee.ipynb), on real data where there is no true answer to compare against.)

It is worth noticing how slowly the error degrades on the right-hand side: over-parameterizing by a factor of four costs a few times the error, not orders of magnitude. That is the `Normal(0, 1)` priors / L2 regularization working --- directions the data do not support are shrunk toward zero rather than left free to fit noise. Erring slightly on the side of too many latents is usually safer than too few.

### Reading the latent space

If we deliberately over-parameterize, can we tell from the fit alone that we have done so? Partly. The singular values of the fitted transformation matrix measure how much each direction of the latent space actually contributes to the predicted data:

In [ ]:
over_model = plx.LVM(latent_size=24)
over_model.register_output("label", LinearTransform(output_size=n_labels))
over_model.register_output("flux", LinearTransform(output_size=n_flux))
over_trained = over_model.optimize_iterative(
    train_data, max_cycles=64, rng_key=jax.random.PRNGKey(112358), progress=False
)

singular_values = np.linalg.svd(
    np.asarray(over_trained.params["flux"]["data"]["A"]), compute_uv=False
)

fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.plot(
    np.arange(1, len(singular_values) + 1),
    singular_values,
    marker="o",
    ls="none",
    color="k",
)
ax.axvline(n_latents + 0.5, color="tab:green", ls="--", lw=1)
_ = ax.set(
    xlabel="latent index",
    ylabel="singular values of $A$",
    yscale="log",
    title="model: 24 latents, data (truth): 8 latents",
)

There is a visible step after eight latent dimensions: the first eight carry most of the structure, and the rest drop by roughly a factor of two and then trail off. So the fit does retain a trace of the true dimensionality. With real data it is usually much subtler. This is mainly a sanity check, not always a useful way to choose the latent size. 

By the way: be careful when trying to interpret these latent dimensions. The individual latent dimensions are not meaningful: as described in the [intro note](LVM-math-notes.ipynb), any rotation of the latent space can be absorbed into the transformation matrices without changing the prediction, so "latent 3" will be something entirely different if you refit from another starting point. What is well-determined is the subspace the latents span. But questions like "how many directions do the data support", "what does motion along this direction of the latent space do to the predicted spectrum", and "are these two stars close together in latent space" are still well posed.

## Preprocessing: why, and when not to

At the start of this tutorial we attached a {py:class}`~pollux.data.ShiftScalePreprocessor` to each output without much justification. It is worth explaining this choice, because the reason is not immediately obvious or well motivated above.

You might think (from machine learning contexts) that preprocessing is necessary to put the outputs on a comparable scale so that no single output dominates the fit. To be clear: That is *not* the reason here. We always optimize or work with a proper likelihood, and rescaling the data and their uncertainties together leaves the likelihood unchanged. 

What preprocessing changes is the prior. A `Normal(0, 1)` prior on the elements of $A$ and on the latents is a statement about *scale*: it says the coefficients should be of order one. In the simulated data above, the first label is a temperature of order 5000 and our second label is an abundance of order 0.1, so in natural units that single prior is a wildly different amount of regularization for the two. Preprocessing puts every output on the scale the default priors assume. We could have instead set the prior scale for each output separately, but for elements of matrices that ends up being a lot of extra bookkeeping.

There is a second, more mechanical reason. By default, {py:class}`~pollux.models.transforms.LinearTransform` computes $A \, z$ with no offset / intercept / bias, so it cannot represent an offset if present in the data. Data with a nonzero mean force the model to spend a latent direction manufacturing one --- or, if the prior will not let it, to simply fail. If it's important to set priors with physical scales, we could use {py:class}`~pollux.models.transforms.AffineTransform` instead to also include an offset.

Both of these effects are easy to see with our simulated data. Here is the same model and the same fit, but now with data we deliberately did not preprocess:

In [ ]:
raw_data = plx.data.PolluxData(
    flux=plx.data.OutputData(data["flux"], err=data["flux_err"]),
    label=plx.data.OutputData(data["label"], err=data["label_err"]),
)
raw_train, raw_test = raw_data[: n_stars // 2], raw_data[n_stars // 2 :]

raw_model = plx.LVM(latent_size=n_latents)
raw_model.register_output("label", LinearTransform(output_size=n_labels))
raw_model.register_output("flux", LinearTransform(output_size=n_flux))

raw_trained = raw_model.optimize_iterative(
    raw_train, max_cycles=64, rng_key=jax.random.PRNGKey(112358), progress=False
)
raw_fixed = raw_model.output_pars(raw_trained.params)
raw_applied = raw_model.optimize_iterative(
    plx.data.PolluxData(flux=raw_test["flux"]),
    blocks=["latents"],
    fixed_pars=raw_fixed,
    progress=False,
)
raw_pred = raw_model.predict_outputs(raw_fixed, latents=raw_applied.params["latents"])

raw_rmse = np.sqrt(
    np.mean(
        (np.asarray(raw_pred["label"]) - truth["label"][n_stars // 2 :]) ** 2, axis=0
    )
)
proc_rmse = np.std(
    predict_test_unprocessed_flux["label"].data - truth["label"][n_stars // 2 :], axis=0
)

print(f"{'':14s} {'Teff [K]':>12s} {'abundance [dex]':>18s}")
print(f"{'preprocessed':14s} {proc_rmse[0]:12.3f} {proc_rmse[1]:18.4f}")
print(f"{'raw':14s} {raw_rmse[0]:12.3f} {raw_rmse[1]:18.4f}")

The model predictions (especially for temperature) fails completely, as we would expect!

As mentioned above, instead of pre-processing the data, we could use an {py:class}`~pollux.models.transforms.AffineTransform` for the labels, which allows the model to learn an offset for each output, and set the scales of the priors in units of the output labels:

In [ ]:
label_scales = np.array([5000.0, 0.1])

affine_model = plx.LVM(latent_size=n_latents)
affine_model.register_output(
    "label",
    AffineTransform(
        output_size=n_labels,
        priors={
            "A": dist.Normal(scale=label_scales.reshape(n_labels, 1)),
            "b": dist.Normal(scale=label_scales),
        },
    ),
)
affine_model.register_output("flux", LinearTransform(output_size=n_flux))

This model now has virtually identical performance to the preprocessed model, and the predictions are now in the natural units of the data:

In [ ]:
affine_trained = affine_model.optimize_iterative(
    raw_train, max_cycles=64, rng_key=jax.random.PRNGKey(112358), progress=False
)
affine_fixed = affine_model.output_pars(affine_trained.params)
affine_applied = affine_model.optimize_iterative(
    plx.data.PolluxData(flux=raw_test["flux"]),
    blocks=["latents"],
    fixed_pars=affine_fixed,
    progress=False,
)
affine_pred = affine_model.predict_outputs(
    affine_fixed, latents=affine_applied.params["latents"]
)

affine_rmse = np.sqrt(
    np.mean(
        (np.asarray(affine_pred["label"]) - truth["label"][n_stars // 2 :]) ** 2, axis=0
    )
)
proc_rmse = np.std(
    predict_test_unprocessed_flux["label"].data - truth["label"][n_stars // 2 :], axis=0
)

print(f"{'':14s} {'Teff [K]':>12s} {'abundance [dex]':>18s}")
print(f"{'preprocessed':14s} {proc_rmse[0]:12.3f} {proc_rmse[1]:18.4f}")
print(f"{'affine':14s} {affine_rmse[0]:12.3f} {affine_rmse[1]:18.4f}")

### When not to preprocess

Preprocessing is not always the right thing to do. Here are some cases where it is particularly bad:

- When the model's structure assumes a scale or a sign: A non-negative model --- e.g., `HalfNormal` priors on the coefficients, or a non-negative matrix factorization --- requires non-negative data, and subtracting the mean destroys that. This is why the [intro note](LVM-math-notes.ipynb) deliberately skips preprocessing for the NMF example. The same applies to a model of the form $1 - W \, H$ for absorption, which assumes continuum-normalized flux near one.
- When the data are already on a sensible scale: Continuum-normalized spectra already have values around one and have a meaningful zero point.
- When some features are nearly constant: Dividing by a per-pixel standard deviation amplifies pixels with little real variation, essentially amplifying noise.

## Conclusion

This tutorial shows how to use the {py:class}`~pollux.models.LVM` class to define a latent variable model with two linear outputs. We have demonstrated how to define the model, optimize it, evaluate it on held-out data, choose the latent dimensionality, and decide whether to preprocess.

Many aspects of the model structure and how the model is used are customizable. The [_Lux_ paper](https://arxiv.org/abs/2502.01745) describes one such model in detail, and {py:class}`~pollux.models.Lux` implements this model structure automatically for you. 

We can construct a very general set of models with the {py:class}`~pollux.models.LVM` class. For example, we could use different (more complex) transformations that map the latent vectors to outputs (e.g., Gaussian process or multi-layer perceptron), or we could use the model in a probabilistic context to perform the train/test application in a single hierarchical inference. We hope to explore these extensions in future tutorials (contributions are welcome!).